<a href="https://colab.research.google.com/github/serahnjogu-new/AI4EAC-Climate-Health-Risk-Prediction-Challenge-/blob/main/AI4EAC_Climate_%26_Health_Risk_Prediction_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier, early_stopping
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.cluster import KMeans

# 1. LOAD & MERGE
train = pd.read_csv('/content/Train.csv')
test = pd.read_csv('/content/Test.csv')
climate = pd.read_csv('/content/climate_features.csv')
ss = pd.read_csv('/content/SampleSubmission.csv')


train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

In [7]:
# 2. SMOOTHED TARGET ENCODING (The Secret to 0.84 AUC)
# This maps the mortality risk of each village without overfitting.
def smoothed_target_encode(train_df, test_df, column, target, weight=10):
    global_mean = train_df[target].mean()

    # Compute counts and means for each category
    agg = train_df.groupby(column)[target].agg(['count', 'mean'])
    counts = agg['count']
    means = agg['mean']

    # Apply smoothing formula
    smooth = (counts * means + weight * global_mean) / (counts + weight)

    train_df[f'{column}_risk'] = train_df[column].map(smooth).fillna(global_mean)
    test_df[f'{column}_risk'] = test_df[column].map(smooth).fillna(global_mean)
    return train_df, test_df

In [8]:
# 3. ADVANCED FEATURE ENGINEERING
def engineer_spatial_climate(df):
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['month'] = df['deathdate'].dt.month

    # Feature 1: Cluster-Relative Temperature
    # Is it hotter than the average for THIS specific village?
    df['temp_anomaly'] = df['tmax_30d'] - df.groupby('location')['tmax_30d'].transform('mean')

    # Feature 2: Rainfall Concentration
    # High rain in few days = Flooding risk (Cholera/Diarrhea)
    df['flood_risk'] = df['rain_sum_30d'] / (df['rain_days_30d'] + 1)

    # Feature 3: Elevation vs Temperature
    # High altitude areas have different climate-mortality thresholds
    df['alt_heat_stress'] = df['elevation'] * df['tavg_30d']

    # Standard Encodings
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone'] = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    return df.drop(['ID', 'deathdate', 'location', 'deathdate_extra'], axis=1, errors='ignore')

# Apply processing
train, test = smoothed_target_encode(train, test, 'location', 'is_climate_sensitive')
X = engineer_spatial_climate(train).drop('is_climate_sensitive', axis=1)
y = train['is_climate_sensitive']
X_test = engineer_spatial_climate(test)

In [9]:
# 4. RANK-OPTIMIZED ENSEMBLE
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oob_probs = np.zeros(len(X))
test_probs = np.zeros(len(X_test))

print("Training Rank-Optimized Ensemble...")

for fold, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    xt, xv = X.iloc[trn_idx], X.iloc[val_idx]
    yt, yv = y.iloc[trn_idx], y.iloc[val_idx]

    # Lower scale_pos_weight to 1.3 to prioritize AUC (ranking) over raw F1 bias
    params = {'learning_rate': 0.01, 'max_depth': 6, 'scale_pos_weight': 1.3, 'random_state': 42}

    lgb = LGBMClassifier(n_estimators=2000, **params)
    cb = CatBoostClassifier(iterations=1500, **params, verbose=False)
    xgb = XGBClassifier(n_estimators=1500, **params, early_stopping_rounds=100, eval_metric='auc')

    lgb.fit(xt, yt, eval_set=[(xv, yv)], callbacks=[early_stopping(100, verbose=False)])
    cb.fit(xt, yt, eval_set=(xv, yv), early_stopping_rounds=100)
    xgb.fit(xt, yt, eval_set=[(xv, yv)], verbose=False)

    # Blend (Giving CatBoost more weight for AUC stability)
    fold_probs = (lgb.predict_proba(xv)[:,1] * 0.25) + (cb.predict_proba(xv)[:,1] * 0.50) + (xgb.predict_proba(xv)[:,1] * 0.25)
    oob_probs[val_idx] = fold_probs
    test_probs += ((lgb.predict_proba(X_test)[:,1] * 0.25) + (cb.predict_proba(X_test)[:,1] * 0.50) + (xgb.predict_proba(X_test)[:,1] * 0.25)) / 5

Training Rank-Optimized Ensemble...
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001502 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5146
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

In [10]:
# 5. FINAL SCORE CHECK
f1 = f1_score(y, (oob_probs >= 0.5).astype(int))
auc = roc_auc_score(y, oob_probs)
print(f"\n--- SCORE IMPROVEMENT ---")
print(f"NEW F1: {f1:.4f} | NEW AUC: {auc:.4f}")
print(f"ESTIMATED TOTAL: {(0.5 * f1) + (0.5 * auc):.4f}")


--- SCORE IMPROVEMENT ---
NEW F1: 0.8156 | NEW AUC: 0.7969
ESTIMATED TOTAL: 0.8063


In [11]:
# 6. EXPORT
pd.DataFrame({'ID': ss['ID'], 'TargetF1': (test_probs >= 0.5).astype(int), 'TargetRAUC': test_probs}).to_csv('084_breakthrough.csv', index=False)

In [12]:
# Create the submission dataframe
submission = pd.DataFrame({
    'ID': ss['ID'], # Uses the ID order from SampleSubmission.csv
    'TargetF1': (test_probs >= 0.5).astype(int),
    'TargetRAUC': test_probs
})

# Save to CSV
submission.to_csv('my_submission.csv', index=False)

In [14]:
from google.colab import files
files.download('my_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import pandas as pd
print(pd.read_csv('my_submission.csv').head())

            ID  TargetF1  TargetRAUC
0  ID_E760D84B         0    0.373953
1  ID_6EDEA907         1    0.666302
2  ID_B9FFC8D8         0    0.489189
3  ID_74C6C94E         0    0.393436
4  ID_0E02825D         1    0.796862
